# 11. 비동기 서브에이전트 — 장기 작업을 배경으로 돌리기

## 학습 목표
- 동기 서브에이전트의 한계(블로킹, 중도 취소 불가)를 이해한다
- `AsyncSubAgent`로 **비차단 배경 작업**을 정의한다
- 슈퍼바이저가 노출받는 5가지 태스크 관리 도구를 사용한다
- ASGI 공동 배포와 HTTP 원격 배포 방식을 구분해 선택한다
- 즉시 폴링 / 태스크 ID 중복 호출 같은 흔한 함정을 피한다

> **preview 기능** — `deepagents>=0.5.0` 한정. API는 변경될 수 있다.

## 선행 요건

비동기 서브에이전트는 **Agent Protocol 서버**(예: `langgraph dev` 로컬 런타임 또는 LangSmith Deployment)가 필요하다. 평범한 Python 세션만으로는 실제 실행이 불가능하다.

- 로컬: `langgraph dev --n-jobs-per-worker 10`
- 원격: LangSmith Deployment (또는 자체 Agent Protocol 서버)

이 노트북은 **정의·설계 가이드**에 초점을 맞춘다. 하단의 `langgraph.json`과 그래프 파일을 스캐폴딩한 뒤 터미널에서 `langgraph dev`로 띄워야 대화가 실제로 동작한다.

In [ ]:
# 환경 설정
from dotenv import load_dotenv
import os

load_dotenv()
assert os.environ.get("ANTHROPIC_API_KEY") or os.environ.get("OPENAI_API_KEY"), \
    "ANTHROPIC_API_KEY 또는 OPENAI_API_KEY가 설정되지 않았습니다!"
print("환경 설정 완료")

---
## 1. 왜 비동기 서브에이전트인가

동기 서브에이전트(`05_subagents.ipynb`)는 **부모가 응답을 받을 때까지 블로킹**된다.

| 시나리오 | 동기 | 비동기 |
|----------|------|--------|
| 10초 이내 작업 | ✅ 적합 | 오버헤드 불필요 |
| 수 분 이상 걸리는 리서치/코드 작업 | ❌ UX 정지 | ✅ 배경 실행 |
| 병렬로 여러 조사 / 코딩 돌리기 | ❌ 순차 | ✅ 동시 실행 |
| 중간에 지시 변경 / 취소 | ❌ 불가 | ✅ 가능 |
| 사용자와 대화하면서 모니터링 | ❌ | ✅ |

비동기 서브에이전트는 **장기 작업 · 병렬 작업 · 중도 조정**이 필요할 때 쓴다. 짧은 작업은 동기 서브에이전트가 더 단순하다.

---
## 2. `AsyncSubAgent` 정의

동기와 달리 **dict이 아닌 `AsyncSubAgent` 클래스**를 쓴다.

| 필드 | 타입 | 필수 | 설명 |
|------|------|------|------|
| `name` | str | ✅ | 고유 식별자. 슈퍼바이저가 태스크를 이 이름으로 발주 |
| `description` | str | ✅ | 무엇을 하는 서브에이전트인지. 슈퍼바이저 위임 판단에 사용 |
| `graph_id` | str | ✅ | Agent Protocol 서버에 등록된 graph/assistant ID. `langgraph.json` 키와 일치 |
| `url` | str | ❌ | 원격 서버 엔드포인트. 생략 시 ASGI 공동배포로 간주 |
| `headers` | dict | ❌ | 원격 서버 인증 헤더 |

In [ ]:
from deepagents import AsyncSubAgent, create_deep_agent

# 공동 배포(ASGI) — url 생략
researcher = AsyncSubAgent(
    name="researcher",
    description="장시간이 걸리는 웹 리서치와 자료 합성을 수행합니다. 깊은 조사가 필요한 주제를 위임하세요.",
    graph_id="researcher",
)

# 원격 HTTP 전송 — url 지정
coder = AsyncSubAgent(
    name="coder",
    description="코드 생성 및 리뷰를 수행합니다. 여러 파일을 수정하는 긴 작업에 적합합니다.",
    graph_id="coder",
    url="https://coder-deployment.langsmith.dev",
)

print(f"researcher -> graph_id={researcher.graph_id}, url={researcher.url or 'ASGI 공동배포'}")
print(f"coder      -> graph_id={coder.graph_id}, url={coder.url}")

---
## 3. 슈퍼바이저 에이전트 만들기

비동기 서브에이전트를 포함한 슈퍼바이저는 일반 `create_deep_agent()`로 만든다. 모델이 **장시간 배경 실행 UX**를 올바르게 유도하도록 시스템 프롬프트에 규칙을 넣는 것이 중요하다.

In [ ]:
SUPERVISOR_PROMPT = """당신은 프로젝트 코디네이터입니다.
장시간이 걸릴 법한 작업은 비동기 서브에이전트에게 위임하고, 즉시 사용자에게 제어를 돌려주세요.

규칙:
- 비동기 작업을 시작하면 **바로 `check_async_task`를 부르지 마세요**. 결과를 모으지 말고 사용자에게 통제권을 반환합니다.
- 사용자가 상태를 물을 때만 `check_async_task` 또는 `list_async_tasks`를 호출합니다.
- `task_id`는 **절대 자르거나 줄이지 마세요**. 항상 전체 ID를 노출합니다.
- 대화 이력에 남은 이전 상태는 stale일 수 있음을 가정하고, 신선한 상태가 필요하면 다시 조회합니다.

한국어로 응답하세요."""

supervisor = create_deep_agent(
    model="anthropic:claude-sonnet-4-6",
    system_prompt=SUPERVISOR_PROMPT,
    subagents=[researcher, coder],
)

print("슈퍼바이저 생성 완료 (async subagents: researcher, coder)")

---
## 4. 슈퍼바이저에게 노출되는 5가지 도구

`AsyncSubAgent`를 등록하면 미들웨어가 슈퍼바이저에게 다음 도구 5종을 자동 주입한다.

| 도구 | 동작 | 언제 호출 |
|------|------|-----------|
| `start_async_task` | 배경 태스크 실행, **즉시 task_id 반환** | 사용자가 긴 작업을 요청할 때 |
| `check_async_task` | 특정 태스크의 현재 상태/결과 조회 | 사용자가 해당 태스크 진행을 물을 때 |
| `update_async_task` | 실행 중인 태스크에 새 지시 전달 | 도중에 범위/방향을 바꿀 때 |
| `cancel_async_task` | 태스크 중단 | 불필요해졌거나 잘못됐을 때 |
| `list_async_tasks` | 모든 추적 태스크와 현재 상태 조회 | 전체 현황 요청 시 |

태스크 메타데이터는 전용 `async_tasks` 채널에 저장되어 **메시지 이력 요약(compaction)을 거쳐도 보존**된다.

---
## 5. `langgraph.json` 스캐폴딩 (ASGI 공동 배포)

`url`을 생략하는 기본 방식은 슈퍼바이저와 서브에이전트를 **같은 LangGraph 서버에 공동 배포**하는 것이다. 모든 그래프를 하나의 `langgraph.json`에 등록한다.

In [ ]:
# 이 셀은 실행 시 프로젝트 루트에 langgraph.json 예시를 생성합니다.
# 실제 프로젝트에서는 상황에 맞게 경로를 조정하세요.
import json
from pathlib import Path

scaffold = {
    "dependencies": ["."],
    "graphs": {
        "supervisor": "./src/supervisor.py:graph",
        "researcher": "./src/researcher.py:graph",
        "coder": "./src/coder.py:graph",
    },
    "env": ".env",
}

scaffold_path = Path("langgraph.example.json")
scaffold_path.write_text(json.dumps(scaffold, indent=2))
print(f"예시 파일 생성: {scaffold_path.resolve()}")
print(json.dumps(scaffold, indent=2))

### 로컬 실행

```bash
# 의존성 설치
uv add deepagents langchain-anthropic langgraph

# 개발 서버 실행 — 워커 풀 여유 있게
langgraph dev --n-jobs-per-worker 10
```

슈퍼바이저 하나 + 동시 서브에이전트 태스크 3개를 굴리려면 최소 4개 슬롯이 필요하다.

### 전송 방식 선택

| 방식 | 사용 | 장점 | 단점 |
|------|------|------|------|
| ASGI (공동 배포) | `url` 생략 | 인프로세스 호출, 지연 없음, 단일 배포 | 서브에이전트 독립 스케일 불가 |
| HTTP (원격) | `url` 지정 | 독립 스케일, 원격 유지보수, 조직 분리 | 네트워크 지연, 인증 필요 |

HTTP 방식에서는 `LANGSMITH_API_KEY` 또는 `LANGGRAPH_API_KEY`로 인증한다.

---
## 6. 전형적인 대화 흐름

사용자: *"AI 에이전트 프레임워크 최신 동향 리서치해 주세요."*

1. 슈퍼바이저가 `start_async_task(name="researcher", instruction=...)` 호출
2. 즉시 `task_id=abc123...` 반환 — **이 시점에 체크하지 않고** 사용자에게 응답
3. 사용자가 다른 질문 진행 가능
4. 잠시 후 사용자: *"리서처 진행 상황?"* → 슈퍼바이저 `check_async_task(task_id="abc123...")` 호출
5. 아직 진행 중이면 상태만 알려주고 종료
6. 완료되면 결과를 요약해 전달

### 중도 조정

- 사용자: *"리서처한테 Korean 기업 사례도 포함하라고 해주세요."* → `update_async_task(task_id=..., instruction="한국 기업 사례 추가")`
- 사용자: *"그만"* → `cancel_async_task(task_id=...)`

---
## 7. 흔한 함정 & 해결

| 함정 | 증상 | 해결 |
|------|------|------|
| **시작 직후 즉시 폴링** | `start_async_task` 다음 셀에 바로 `check_async_task` 호출 → UX 정지 | 시스템 프롬프트에 "시작 후 항상 사용자에게 제어 반환" 명시 (위 `SUPERVISOR_PROMPT` 참조) |
| **태스크 ID 잘림** | 모델이 `abc12...` 같이 축약해서 저장 → 이후 조회 실패 | 시스템 프롬프트에 "task_id는 절대 자르지 말고 항상 전체 표시" 명시 |
| **stale 상태 보고** | 과거 대화에서 본 상태를 최신인 것처럼 말함 | 미들웨어가 stale임을 모델에 알려주지만, 사용자가 물으면 반드시 `check_async_task` 다시 호출 |
| **워커 부족** | 여러 태스크가 대기 상태로 적체 | `langgraph dev --n-jobs-per-worker` 값을 올리거나 배포 환경 워커 수 증설 |

---

## 8. 언제 동기 vs 비동기

| 상황 | 선택 |
|------|------|
| 간단한 위임, 10초 이내 | 동기 (`SubAgent` dict) |
| 수 분 이상 리서치·코딩·빌드 | **비동기** |
| 병렬로 여러 작업 | **비동기** |
| 사용자와 대화하며 중도 수정/취소 | **비동기** |
| 같은 세션 내 여러 툴 사용만 필요 | 동기 |
| 별도 서비스로 분리 필요 | 비동기 + HTTP `url` |

---
## 핵심 정리

| 항목 | 내용 |
|------|------|
| 클래스 | `AsyncSubAgent(name, description, graph_id, url?, headers?)` |
| 전송 방식 | ASGI 공동 배포(기본) / HTTP 원격(`url` 지정) |
| 슈퍼바이저 도구 | `start/check/update/cancel/list_async_task` |
| 상태 저장 | 전용 `async_tasks` 채널 (compaction 생존) |
| 런타임 | `langgraph dev` 또는 LangSmith Deployment |
| 시스템 프롬프트 필수 규칙 | 즉시 폴링 금지, task_id 자르지 말기, stale 가정 |
| 상태 | preview (deepagents 0.5.0+) — API 변경 가능 |

## 다음 단계
→ **LangSmith Studio로 비동기 태스크 모니터링** — 장기 작업 디버깅은 Studio UI가 가장 편리합니다.
→ **`10_sandboxes_and_acp.ipynb`**: 비동기 서브에이전트 내부에서 샌드박스를 함께 쓰는 구성.